In [1]:
import xarray as xr, numpy as np, pandas as pd
import rioxarray as rxr

In [6]:
from pathlib import Path
data = Path("/Users/ridhoain/Canopy LCZ Project/")

In [ ]:
datas = xr.open_dataset(data/"era5land.nc")
datas

In [ ]:
# File patterns (glob-style)
PAT_HIST = {
    "tmean": "t2m_java_hist_proj/hist_tas/*.nc",  # one monthly file per year
    "tmin" : "t2m_java_hist_proj/hist_tasmin/*.nc",
    "tmax" : "t2m_java_hist_proj/hist_tasmax/*.nc",
}

# Projection files are organized per scenario (each with one monthly file per year)
SCENARIOS = ["ssp126", "ssp245", "ssp585"]  # adjust to what you have
PAT_PROJ = {
    "tmean": {scen: f"t2m_java_hist_proj/proj_{scen}_tas/*.nc" for scen in SCENARIOS},
    "tmin" : {scen: f"t2m_java_hist_proj/proj_{scen}_tasmin/*.nc"  for scen in SCENARIOS},
    "tmax" : {scen: f"t2m_java_hist_proj/proj_{scen}_tasmax/*.nc"  for scen in SCENARIOS},
}

# LCZ raster (GeoTIFF or similar). Classes: 1–10 built, 11–17 natural (A–G)
LCZ_PATH = "LCZ_Subset_Java.tif"

# Baseline years (used to select historical files if your hist span is larger)
BASELINE_START, BASELINE_END = 2010, 2014  # IPCC-ish 20-yr baseline

# Define your "decades" for projections (inclusive)
DECADES = {
    "2030s": (2030, 2039),
    "2050s": (2050, 2059),
    "2080s": (2080, 2089),
}

# LCZ groups you care about
LCZ_BUILT   = [1,2,3,4,5,6]
LCZ_PERI    = [5,6]
LCZ_NATURAL = [11,12,13,14]


In [3]:
def to_celsius(da):
    """
    Convert K -> °C when needed.
    Tries units attr first; falls back to a small computed sample heuristic.
    Works with dask-backed arrays.
    """
    units = (da.attrs.get("units", "") or "").lower()
    if units in {"k", "kelvin"}:
        return da - 273.15
    if units in {"c", "degc", "°c", "celsius", "degree_celsius"}:
        return da

    # Heuristic: compute a tiny sample mean (avoids loading the whole array)
    try:
        # pick first along each dim to keep it tiny
        indexers = {d: 0 for d in da.dims if da.sizes.get(d, 0) > 0}
        sample = da.isel(**indexers).mean().compute().item()
    except Exception:
        # fallback: this may load more; only used if compute() path fails
        import numpy as np
        sample = float(np.nanmean(da.values))

    return da - 273.15 if sample > 200 else da


In [4]:

# --------------------
# HELPERS
# --------------------
def open_monthly_series(pattern, varname=None, year_filter=None, chunks=None):
    """Open monthly NetCDFs (one file per year) into one Dataset."""
    ds = xr.open_mfdataset(pattern, combine="by_coords", chunks=chunks)
    # Try to pick a sensible var if not given
    if varname is None:
        # prefer standard names if present
        for cand in ["tmean","tmin","tmax","tas","tasmin","tasmax"]:
            if cand in ds.data_vars:
                varname = cand; break
        else:
            varname = list(ds.data_vars)[0]
    da = ds[varname]
    # Ensure a proper time coordinate
    if "time" not in da.dims:
        raise ValueError("Expected a time dimension in monthly files.")
    # Filter years if requested
    if year_filter is not None:
        years = xr.DataArray(da["time"].dt.year)
        da = da.sel(time=slice(str(year_filter[0]), str(year_filter[1])))
    return to_celsius(da)

def monthly_clim_12(da):
    """12-month climatology: average all years by calendar month."""
    return da.groupby("time.month").mean("time", skipna=True)

def monthly_clim_12_decadal(da, decades_dict):
    """
    Return 12-month climatologies per decade label.
    Output dims: decade, month, y, x (and scenario if da has it).
    """
    out = []
    for dec_label, (y0,y1) in decades_dict.items():
        sub = da.sel(time=slice(str(y0), str(y1)))
        if sub.time.size == 0:
            continue
        clim = sub.groupby("time.month").mean("time", skipna=True)
        clim = clim.assign_coords(decade=dec_label).expand_dims("decade")
        out.append(clim)
    return xr.concat(out, dim="decade") if out else None

def lcz_to_model_grid(lcz_path, ref_da):
    """Read LCZ raster and snap to model grid (nearest)."""
    lcz = rxr.open_rasterio(lcz_path, masked=True).squeeze()
    # If LCZ has no CRS, assume EPSG:4326 (adjust if yours differs)
    if lcz.rio.crs is None:
        lcz = lcz.rio.write_crs("EPSG:4326")
    # If model grid has no CRS, try to attach WGS84
    if not ref_da.rio.crs:
        ref_da = ref_da.rio.write_crs("EPSG:4326")
    lcz_on_model = lcz.rio.reproject(ref_da.rio.crs, resampling=rxr.rio.reproject.NEAREST)
    lcz_on_model = lcz_on_model.rio.reproject_match(ref_da, resampling=rxr.rio.reproject.NEAREST)
    return lcz_on_model

def group_mean_by_lcz(da, lcz_da, groups_dict):
    """
    LCZ group means for each time-like coordinate in da (e.g., month or (decade, month)).
    Returns a tidy DataFrame.
    """
    frames = []
    for gname, classes in groups_dict.items():
        mask = xr.where(np.isin(lcz_da, classes), 1.0, np.nan)
        gm = (da * mask).mean(("y","x"), skipna=True)  # dims: (..., month)
        df = gm.to_dataframe(name="value").reset_index()
        df["group"] = gname
        frames.append(df)
    return pd.concat(frames, ignore_index=True)


In [5]:
# LOAD & PREP
# --------------------
# Open historical monthly series (per variable), restricted to baseline years
hist = {}
for var, pat in PAT_HIST.items():
    hist[var] = open_monthly_series(pat, varname=None, year_filter=(BASELINE_START, BASELINE_END),
                                    chunks={"time": 12, "y": 200, "x": 200})

# Build 12-month baseline climatologies
base_12 = {var: monthly_clim_12(da) for var, da in hist.items()}  # dims: month,y,x

# Use tmean baseline as a reference for grid/CRS to place LCZ
ref_for_lcz = base_12["tmean"].isel(month=0)
lcz_grid = lcz_to_model_grid(LCZ_PATH, ref_for_lcz)


In [ ]:
# PROJECTIONS
# --------------------
# For each scenario, open all yearly files and compute decadal 12-month climatologies
proj_12 = {}  # proj_12[var][scenario] -> (decade, month, y, x)
for var, scen_map in PAT_PROJ.items():
    proj_12[var] = {}
    for scen, pat in scen_map.items():
        da = open_monthly_series(pat, varname=None, year_filter=None, chunks={"time": 12, "y": 200, "x": 200})
        clim_dec = monthly_clim_12_decadal(da, DECADES)
        if clim_dec is not None:
            proj_12[var][scen] = clim_dec  # dims: decade, month, y, x

# --------------------
# DELTAS (future - baseline) per scenario & decade
# --------------------
delta_12 = {}  # delta_12[var][scenario] -> (decade, month, y, x)
for var in ["tmean", "tmin", "tmax"]:
    delta_12[var] = {}
    base = base_12[var]  # (month,y,x)
    for scen, fut in proj_12[var].items():
        # broadcast base across decade
        base_b = base.broadcast_like(fut)
        delta_12[var][scen] = fut - base_b


In [ ]:
# LCZ GROUP SUMMARIES
# --------------------
GROUPS = {
    "BUILT":   LCZ_BUILT,
    "PERI_5_6": LCZ_PERI,
    "NATURAL_11_14": LCZ_NATURAL
}

# Baseline (per variable)
df_base = []
for var, base in base_12.items():
    df = group_mean_by_lcz(base, lcz_grid, GROUPS)
    df["what"] = "baseline"
    df["var"] = var
    df_base.append(df)
df_base = pd.concat(df_base, ignore_index=True)

# Future and deltas (per variable, per scenario, per decade)
df_fut = []
df_delta = []
for var, scen_map in proj_12.items():
    for scen, fut in scen_map.items():
        dff = group_mean_by_lcz(fut, lcz_grid, GROUPS)
        dff["what"] = "future"
        dff["var"] = var
        dff["scenario"] = scen
        df_fut.append(dff)
        # delta
        dfd = group_mean_by_lcz(delta_12[var][scen], lcz_grid, GROUPS)
        dfd["what"] = "delta"
        dfd["var"] = var
        dfd["scenario"] = scen
        df_delta.append(dfd)

df_fut = pd.concat(df_fut, ignore_index=True) if df_fut else pd.DataFrame()
df_delta = pd.concat(df_delta, ignore_index=True) if df_delta else pd.DataFrame()


In [ ]:
# URBAN–NATURAL CONTRAST (baseline & future)
# --------------------
def contrast_from_tidy(df, var, filt=None):
    """Compute BUILT - NATURAL contrast by (month[, decade, scenario])."""
    q = df.query(f"var == '{var}'")
    if filt:
        q = q.query(filt)
    # pivot to have groups as columns
    piv = q.pivot_table(index=[c for c in q.columns if c in ["month","decade","scenario"]],
                        columns="group", values="value")
    # ensure required groups present
    for needed in ["BUILT","NATURAL_11_14"]:
        if needed not in piv.columns:
            piv[needed] = np.nan
    out = (piv["BUILT"] - piv["NATURAL_11_14"]).reset_index(name="contrast_C")
    out["var"] = var
    return out

contrast_base = contrast_from_tidy(df_base, var="tmax")  # you can also compute for tmean/tmin
contrast_fut_list = []
for scen in SCENARIOS:
    if df_fut.empty: break
    cf = contrast_from_tidy(df_fut, var="tmax", filt=f"scenario == '{scen}'")
    contrast_fut_list.append(cf.assign(scenario=scen))
contrast_fut = pd.concat(contrast_fut_list, ignore_index=True) if contrast_fut_list else pd.DataFrame()


In [ ]:
# EXAMPLE SAVES (optional)
# --------------------
df_base.to_csv("baseline_LCZ_monthly_tmean_tmin_tmax.csv", index=False)
df_fut.to_csv("future_LCZ_monthly_tmean_tmin_tmax.csv", index=False)
df_delta.to_csv("delta_LCZ_monthly_tmean_tmin_tmax.csv", index=False)
contrast_base.to_csv("contrast_baseline_tmax_BUILT_vs_NATURAL.csv", index=False)
if not contrast_fut.empty:
    contrast_fut.to_csv("contrast_future_tmax_BUILT_vs_NATURAL.csv", index=False)

print("Done. Wrote CSVs for baseline, future, deltas, and contrasts.")
